In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_FILE = Path("data/raw/Pokémon pricing data.xlsx")
INTERIM_DIR = Path("data/interim")
PROCESSED_DIR = Path("data/processed")
TABLES_DIR = Path("outputs/tables")

for folder in [INTERIM_DIR, PROCESSED_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

In [2]:
df = pd.read_excel(DATA_FILE, sheet_name="Card List", header=1, dtype={"Card Number": str})

df.columns = [re.sub(r"\s+", " ", str(col).strip()) for col in df.columns]
df = df.dropna(how="all").dropna(axis=1, how="all")

drop_cols = [
    "Owned", "Default Order", "Code Duplicate All",
    "Code Duplicate Owned", "Card ID"
]
unnamed = [col for col in df.columns if col.lower().startswith("unnamed") or col.lower() == "nan"]
df = df.drop(columns=drop_cols + unnamed, errors="ignore")
df = df.loc[:, ~df.columns.duplicated()].copy()

display(df.head())
print(df.shape)

,Collection,Set Name,Card Number,Card Name,Rarity,Card Type,Pokémon Type,HP,Illustrator,Value
0,1st Edition,Base Set,1,Alakazam,★H,Stage 2,Psychic,80,Ken Sugimori,799.99
1,1st Edition,Base Set,2,Blastoise,★H,Stage 2,Water,100,Ken Sugimori,1759.99
2,1st Edition,Base Set,3,Chansey,★H,Basic,Colorless,120,Ken Sugimori,419.99
3,1st Edition,Base Set,4,Charizard,★H,Stage 2,Fire,120,Mitsuhiro Arita,7999.99
4,1st Edition,Base Set,5,Clefairy,★H,Basic,Colorless,40,Ken Sugimori,439.99


(40483, 10)


In [3]:
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip().replace({
        "": pd.NA, "nan": pd.NA, "None": pd.NA,
        "N/A": pd.NA, "NA": pd.NA, "-": pd.NA
    })

def to_number(series):
    cleaned = (
        series.astype("string")
        .str.replace(r"[,£$€]", "", regex=True)
        .str.extract(r"([-+]?\d*\.?\d+)", expand=False)
    )
    return pd.to_numeric(cleaned, errors="coerce")

for column in ["Value", "HP"]:
    df[column] = to_number(df[column])
df["Card Number"] = df["Card Number"].astype("string").str.strip()

duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

In [4]:
has_price = df["Value"].notna() & (df["Value"] > 0)
priced = df[has_price].copy()
unpriced = df[~has_price].copy()
priced["Log_Value"] = np.log1p(priced["Value"])

summary = pd.DataFrame([{
    "raw_rows": len(df) + duplicates_removed,
    "cleaned_rows": len(df),
    "priced_rows": len(priced),
    "unpriced_rows": len(unpriced),
    "duplicates_removed": duplicates_removed,
    "median_value": priced["Value"].median(),
    "mean_value": priced["Value"].mean(),
    "maximum_value": priced["Value"].max()
}])

missing = df.isna().sum().rename("missing_count").to_frame()
missing["missing_pct"] = missing["missing_count"] / len(df) * 100
missing = missing.sort_values("missing_pct", ascending=False)

display(summary.T)
display(missing.head(20))

,0
raw_rows,40483.000000
cleaned_rows,40455.000000
priced_rows,35732.000000
unpriced_rows,4723.000000
duplicates_removed,28.000000
median_value,0.620000
mean_value,10.022121
maximum_value,7999.990000


,missing_count,missing_pct
HP,7997,19.767643
Illustrator,7,0.017303
Set Name,0,0.000000
Collection,0,0.000000
Card Number,0,0.000000
Card Name,0,0.000000
Card Type,0,0.000000
Rarity,0,0.000000
Pokémon Type,0,0.000000
Value,0,0.000000


In [5]:
df.to_csv(INTERIM_DIR / "pokemon_cleaned_full.csv", index=False, encoding="utf-8-sig")
priced.to_csv(PROCESSED_DIR / "pokemon_priced_cards.csv", index=False, encoding="utf-8-sig")
unpriced.to_csv(PROCESSED_DIR / "pokemon_unpriced_cards.csv", index=False, encoding="utf-8-sig")
summary.to_csv(TABLES_DIR / "cleaning_summary.csv", index=False, encoding="utf-8-sig")
missing.to_csv(TABLES_DIR / "missing_value_profile.csv", encoding="utf-8-sig")

print("Priced:", priced.shape)
print("Unpriced:", unpriced.shape)

Priced: (35732, 11)
Unpriced: (4723, 10)
